## FT DDP embeddings -> XGBoost (two-step, no DDP wait)

This example avoids mixing DDP and non-DDP trainers in the same torchrun job:
1) Run FTTransformer in `unsupervised_embedding` mode with DDP and cache embeddings.
2) Merge cached embeddings back into the raw dataset.
3) Train XGBoost in a single process on the augmented data.

Files used:
- Step 1 config: `examples/modelling/config_ft_unsupervised_ddp_embed.json`
- Cached embeddings: `<output_dir>/Results/predictions/<model>_ft_emb_train.csv` / `_test.csv`
- Step 2 config will be generated as `examples/modelling/config_xgb(resn)_from_ft_unsupervised.json`


In [1]:
from ins_pricing.cli.Pricing_Run import run
import copy
import json
from pathlib import Path

import pandas as pd

from ins_pricing.cli.utils.cli_common import split_train_test

In [ ]:
# Step 1: FT unsupervised embedding with DDP (nproc_per_node=2)
run("config_ft_unsupervised_ddp_embed.json")

In [ ]:
# Work in current directory - use relative paths
work_dir = Path.cwd()

cfg_path = work_dir / "config_ft_unsupervised_ddp_embed.json"
cfg = json.loads(cfg_path.read_text(encoding="utf-8"))

model_name = f"{cfg['model_list'][0]}_{cfg['model_categories'][0]}"
data_dir = (cfg_path.parent / cfg["data_dir"]).resolve()
raw_path = data_dir / f"{model_name}.csv"
raw = pd.read_csv(raw_path)

holdout_ratio = cfg.get("holdout_ratio", cfg.get("prop_test", 0.25))
split_strategy = cfg.get("split_strategy", "random")
split_group_col = cfg.get("split_group_col")
split_time_col = cfg.get("split_time_col")
split_time_ascending = cfg.get("split_time_ascending", True)
rand_seed = cfg.get("rand_seed", 13)

train_df, test_df = split_train_test(
    raw,
    holdout_ratio=holdout_ratio,
    strategy=split_strategy,
    group_col=split_group_col,
    time_col=split_time_col,
    time_ascending=split_time_ascending,
    rand_seed=rand_seed,
    reset_index_mode="time_group",
    ratio_label="holdout_ratio",
)

out_root = (cfg_path.parent / cfg["output_dir"]).resolve()
pred_prefix = cfg.get("ft_feature_prefix", "ft_emb")
pred_dir = out_root / "Results" / "predictions"

train_pred_path = pred_dir / f"{model_name}_{pred_prefix}_train.csv"
test_pred_path = pred_dir / f"{model_name}_{pred_prefix}_test.csv"

pred_train = pd.read_csv(train_pred_path)
pred_test = pd.read_csv(test_pred_path)

if len(pred_train) != len(train_df) or len(pred_test) != len(test_df):
    raise ValueError(
        "Prediction rows do not match split sizes; check split settings.")

# Attach embeddings back to the original rows using the split indices.
aug = raw.copy()
aug.loc[train_df.index, pred_train.columns] = pred_train.values
aug.loc[test_df.index, pred_test.columns] = pred_test.values

data_out_dir = cfg_path.parent / "DataFTUnsupervised"
data_out_dir.mkdir(parents=True, exist_ok=True)
aug_path = data_out_dir / f"{model_name}.csv"
aug.to_csv(aug_path, index=False)
print(f"Augmented data saved to: {aug_path}")

In [ ]:

# Build a single-process XGB config that includes embedding columns.
embed_cols = list(pred_train.columns)
xgb_cfg = copy.deepcopy(cfg)
xgb_cfg["data_dir"] = str(data_out_dir.relative_to(cfg_path.parent))
xgb_cfg["feature_list"] = cfg["feature_list"] + embed_cols
xgb_cfg["ft_role"] = "model"
xgb_cfg["stack_model_keys"] = ["xgb"]
xgb_cfg["cache_predictions"] = False
xgb_cfg["use_resn_ddp"] = False
xgb_cfg["use_ft_ddp"] = False
xgb_cfg["use_gnn_ddp"] = False
xgb_cfg["use_resn_data_parallel"] = False
xgb_cfg["use_ft_data_parallel"] = False
xgb_cfg["use_gnn_data_parallel"] = False
xgb_cfg["output_dir"] = "./ResultsXGBFromFTUnsupervised"
xgb_cfg["optuna_storage"] = "./ResultsXGBFromFTUnsupervised/optuna/bayesopt.sqlite3"
xgb_cfg["optuna_study_prefix"] = "pricing_ft_unsup_xgb"

runner_cfg = xgb_cfg.get("runner", {}) or {}
runner_cfg["model_keys"] = ["xgb"]
runner_cfg["nproc_per_node"] = 1
runner_cfg["plot_curves"] = False
xgb_cfg["runner"] = runner_cfg

xgb_cfg["plot_curves"] = False
plot_cfg = xgb_cfg.get("plot", {}) or {}
plot_cfg["enable"] = False
xgb_cfg["plot"] = plot_cfg

xgb_cfg_path = cfg_path.parent / "config_xgb_from_ft_unsupervised.json"
xgb_cfg_path.write_text(json.dumps(xgb_cfg, indent=2))
print(f"Wrote XGB config: {xgb_cfg_path}")

In [ ]:
# Build a DDP ResNet config that includes embedding columns.
embed_cols = list(pred_train.columns)
resn_cfg = copy.deepcopy(cfg)
resn_cfg["data_dir"] = str(data_out_dir.relative_to(cfg_path.parent))
resn_cfg["feature_list"] = cfg["feature_list"] + embed_cols
resn_cfg["ft_role"] = "model"
resn_cfg["stack_model_keys"] = ["resn"]
resn_cfg["cache_predictions"] = False
resn_cfg["use_resn_ddp"] = True
resn_cfg["use_ft_ddp"] = False
resn_cfg["use_gnn_ddp"] = False
resn_cfg["use_resn_data_parallel"] = False
resn_cfg["use_ft_data_parallel"] = False
resn_cfg["use_gnn_data_parallel"] = False
resn_cfg["output_dir"] = "./ResultsResNFromFTUnsupervised"
resn_cfg["optuna_storage"] = "./ResultsResNFromFTUnsupervised/optuna/bayesopt.sqlite3"
resn_cfg["optuna_study_prefix"] = "pricing_ft_unsup_resn_ddp"

runner_cfg = resn_cfg.get("runner", {}) or {}
runner_cfg["model_keys"] = ["resn"]
runner_cfg["nproc_per_node"] = 2
runner_cfg["plot_curves"] = False
resn_cfg["runner"] = runner_cfg

resn_cfg["plot_curves"] = False
plot_cfg = resn_cfg.get("plot", {}) or {}
plot_cfg["enable"] = False
resn_cfg["plot"] = plot_cfg

resn_cfg_path = cfg_path.parent / "config_resn_from_ft_unsupervised.json"
resn_cfg_path.write_text(json.dumps(resn_cfg, indent=2))
print(f"Wrote ResNet config: {resn_cfg_path}")

In [ ]:

# Step 2: XGBoost on the augmented dataset (single process, non-DDP)
run("config_xgb_from_ft_unsupervised.json")

In [ ]:
# Step 2: ResNet with DDP in a new process
run("config_resn_from_ft_unsupervised.json")